# PS2 episode-closure diagnostic

Read-only. This notebook does not write to S3, does not touch the Gold layer, does not
publish, and does not modify the PS2 notebook or any PS2 output.

**What it answers.** PS2 reports recorded hardware-OOS time as a very large share of
available device-hours -- highest for the validator fleet, where it approaches the whole
window. That is not physically credible. Either the fleet is genuinely down that much, or
episodes open and never close in the source data and the duration is an artefact.

**How.** It borrows PS2's own setup cells (0-4) so the source paths, the fleet mapping and
the column contracts cannot drift, then runs three independent checks:

- **A** `direct_s3_silver.device_event_enriched` -- is every hardware-OOS `Set` followed by
  a `Clear`, both as a per-fleet ratio and per row via `CLEAR_DTM`?
- **B** `mars_dev.silver.device_outage` -- the table that actually produces
  `hardware_oos_minutes`. How many episodes have no `outage_end`, and how long are they?
- **C** rebuild the share of available device-hours, then remove episodes longer than a
  day and see how much survives.

**Standing caveat.** The dump ends 11 April 2026. An episode still open at that boundary is
right-censored by construction, not by a defect. C separates the two rather than blaming
the cutoff for everything. A missing tail can only make a closed window look shorter, never
longer, so it cannot by itself explain an inflated share.

**To run.** Set `PS2_NOTEBOOK_PATH` if your repo clone is not at
`~/Chicago-Ventra-Mars-Cubic-Analysis`, then Run All. Only PS2 cells 0-4 are executed;
the expensive PS2 load and publish cells are not.


In [ ]:
# =============================================================================
# PS2 EPISODE-CLOSURE DIAGNOSTIC -- CELL 1 of 2: borrow the PS2 setup, change nothing
#
# READ-ONLY. This notebook never writes to S3, never touches Gold, never publishes,
# and never modifies the PS2 notebook. It reuses PS2's OWN setup cells so the source
# paths, the category mapping and the column contracts cannot drift from the pipeline
# they are meant to explain.
#
# It execs code cells 0..4 of the PS2 notebook. Those cells are pure setup:
#   0 run id     1 runtime bootstrap     2 configuration and the SOURCE dict
#   3 Spark + Delta + S3A session        4 reader/contract function definitions
# Cell 4 only DEFINES read_source(); it performs no Spark action. The expensive
# loads (PS2 cell 5 onward) are deliberately NOT run.
#
# RUN_MODE is forced to REPLAY on an isolated export prefix before the exec, so even
# if a later PS2 cell were run by accident it could not reach the production
# dispatcher prefix that the RDS loader watches.
# =============================================================================
import os, json
from pathlib import Path

os.environ['PS2_RUN_MODE'] = 'REPLAY'
os.environ['PS2_EXPORT_PREFIX'] = 'ps2_diagnostic_setclear_outputs'
os.environ.setdefault('PS2_MAX_SOURCE_LAG_DAYS', '150')

PS2_NOTEBOOK = Path(os.getenv(
    'PS2_NOTEBOOK_PATH',
    Path.home() / 'Chicago-Ventra-Mars-Cubic-Analysis' / 'notebooks'
    / 'ps2_cascading_failure' / 'PS2_Failure_Patterns_v2_5_3_Storage_Safe.ipynb',
))
if not PS2_NOTEBOOK.exists():
    raise FileNotFoundError(
        f'Cannot find the PS2 notebook at {PS2_NOTEBOOK}. Set PS2_NOTEBOOK_PATH to its '
        'location in the synced repo clone.'
    )

_nb = json.loads(PS2_NOTEBOOK.read_text(encoding='utf-8'))
_code = [''.join(c['source']) for c in _nb['cells'] if c['cell_type'] == 'code']
SETUP_CELLS = 5
if len(_code) < SETUP_CELLS:
    raise RuntimeError(f'Expected at least {SETUP_CELLS} code cells in PS2, found {len(_code)}.')

# Refuse to exec anything that writes. If PS2's setup cells ever grow a publish call,
# this stops rather than silently doing it.
_forbidden = ('put_object', 'upload_file', 'saveAsTable', '.write.', 'delete_object')
for _i in range(SETUP_CELLS):
    _hit = [t for t in _forbidden if t in _code[_i]]
    if _hit:
        raise RuntimeError(
            f'PS2 setup cell {_i} now contains a write operation {_hit}; this diagnostic '
            'will not exec it. Review the notebook before continuing.'
        )

for _i in range(SETUP_CELLS):
    print(f'--- exec PS2 code cell {_i} ({len(_code[_i]):,} chars)')
    exec(compile(_code[_i], f'<PS2 code cell {_i}>', 'exec'), globals())

for _needed in ('spark', 'read_source', 'canonical_device_category', 'truthy',
                'TARGET_DEVICE_CATEGORIES', 'ANALYSIS_LOOKBACK_DAYS', 'SOURCE'):
    if _needed not in globals():
        raise RuntimeError(f'PS2 setup did not provide {_needed!r}; cannot continue.')

print()
print('Setup borrowed from PS2. No PS2 load or publish cell was run.')
print('label_events ->', SOURCE['label_events']['path'])
print('outage       ->', SOURCE['outage']['path'])


In [ ]:
# =============================================================================
# PS2 EPISODE-CLOSURE DIAGNOSTIC -- CELL 2 of 2: does OOS time actually end?
#
# THE QUESTION. PS2 reports recorded hardware-OOS time as a very large share of
# available device-hours, highest for VALIDATOR. A share near 100% is not physically
# credible, so either the fleet really is down that much, or episodes are opening and
# never closing in the data and the duration is an artefact.
#
# WHERE THE NUMBER ACTUALLY COMES FROM. PS2's hardware_oos_minutes is
# sum(device_outage.duration_min); its onsets are countDistinct(outage_episode_id).
# The Set/Clear event state only defines the LABEL
# (is_hardware_oos_event = true AND EVENT_STATE_TYPE_NAME = 'Set').
# So this cell tests BOTH independently:
#
#   PART A  device_event_enriched -- is every Set followed by a Clear?
#   PART B  device_outage         -- do episodes have an end, and how long are they?
#   PART C  the reconstruction    -- rebuild the share and see what carries it.
#
# READ-ONLY. Nothing here writes.
#
# CAVEAT THAT APPLIES TO EVERY NUMBER BELOW. The dump ends 11 April 2026. An episode
# still open at the boundary is right-censored by construction, not by a data defect.
# The cell separates censored-at-boundary from genuinely-unclosed for that reason.
# =============================================================================
import datetime as dt
import pandas as pd
from pyspark.sql import functions as F

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

TARGETS = list(TARGET_DEVICE_CATEGORIES)

# ---------------------------------------------------------------- PART A: events
ev = read_source(
    'label_events',
    ['DEVICE_ID', 'EVENT_DTM', 'EVENT_STATE_TYPE_NAME', 'is_hardware_oos_event',
     'mars_device_category'],
    ['CLEAR_DTM', 'is_commanded_oos_event', 'is_oos_event', 'DW_DEVICE_EVENT_ID'],
)
HAS_CLEAR_DTM = 'CLEAR_DTM' in ev.columns
print('columns projected from the direct-Silver event export:', ev.columns)
print('CLEAR_DTM available:', HAS_CLEAR_DTM)
print()

ev = (
    ev
    .withColumn('fleet', canonical_device_category(F.col('mars_device_category')))
    .withColumn('event_ts', F.col('EVENT_DTM').cast('timestamp'))
    .withColumn('state', F.upper(F.trim(F.col('EVENT_STATE_TYPE_NAME').cast('string'))))
    .where(F.col('fleet').isin(*TARGETS))
    .where(truthy(F.col('is_hardware_oos_event')))
)

AS_OF_TS = ev.agg(F.max('event_ts').alias('m')).first()['m']
if AS_OF_TS is None:
    raise RuntimeError('No hardware-OOS events in the target fleets; nothing to diagnose.')
WIN_END = AS_OF_TS
WIN_START = WIN_END - dt.timedelta(days=ANALYSIS_LOOKBACK_DAYS)
WIN_DAYS = ANALYSIS_LOOKBACK_DAYS
print(f'Window: {WIN_START} -> {WIN_END}  ({WIN_DAYS} days, PS2 ANALYSIS_LOOKBACK_DAYS)')
print()

evw = ev.where((F.col('event_ts') >= F.lit(WIN_START)) & (F.col('event_ts') <= F.lit(WIN_END)))

# A.1 -- what state values actually exist. Do not assume the vocabulary is Set/Clear.
print('A.1  EVENT_STATE_TYPE_NAME vocabulary on hardware-OOS events')
state_vocab = (
    evw.groupBy('state').agg(F.count('*').alias('events'))
    .orderBy(F.desc('events')).toPandas()
)
print(state_vocab.to_string(index=False))
print()

# A.2 -- Set vs Clear per fleet. A healthy log has roughly one Clear per Set.
print('A.2  Set vs Clear per fleet (the headline test)')
per_fleet = (
    evw.groupBy('fleet')
    .agg(
        F.sum(F.when(F.col('state') == 'SET', 1).otherwise(0)).alias('n_set'),
        F.sum(F.when(F.col('state') == 'CLEAR', 1).otherwise(0)).alias('n_clear'),
        F.sum(F.when(~F.col('state').isin('SET', 'CLEAR'), 1).otherwise(0)).alias('n_other_state'),
        F.countDistinct('DEVICE_ID').alias('devices'),
    )
    .toPandas()
)
per_fleet['clear_per_set'] = (per_fleet['n_clear'] / per_fleet['n_set'].clip(lower=1)).round(4)
per_fleet['unmatched_sets'] = per_fleet['n_set'] - per_fleet['n_clear']
print(per_fleet.sort_values('fleet').to_string(index=False))
print()

# A.3 -- the same question asked of the row itself, which is stronger than a count
# ratio because it pairs Set to Clear per event rather than in aggregate.
if HAS_CLEAR_DTM:
    print('A.3  Of the Set rows, how many carry a CLEAR_DTM')
    sets = (
        evw.where(F.col('state') == 'SET')
        .withColumn('clear_ts', F.col('CLEAR_DTM').cast('timestamp'))
        .withColumn(
            'open_minutes',
            F.when(
                F.col('clear_ts').isNotNull(),
                (F.col('clear_ts').cast('long') - F.col('event_ts').cast('long')) / 60.0,
            ),
        )
    )
    closure = (
        sets.groupBy('fleet')
        .agg(
            F.count('*').alias('n_set'),
            F.sum(F.when(F.col('clear_ts').isNotNull(), 1).otherwise(0)).alias('with_clear_dtm'),
            F.sum(F.when(F.col('clear_ts').isNull(), 1).otherwise(0)).alias('null_clear_dtm'),
            F.sum(F.when(F.col('clear_ts') > F.lit(WIN_END), 1).otherwise(0)).alias('clear_after_asof'),
            F.sum(F.when(F.col('open_minutes') < 0, 1).otherwise(0)).alias('clear_before_set'),
            F.expr('percentile_approx(open_minutes, 0.5)').alias('p50_open_min'),
            F.expr('percentile_approx(open_minutes, 0.9)').alias('p90_open_min'),
            F.expr('percentile_approx(open_minutes, 0.99)').alias('p99_open_min'),
            F.max('open_minutes').alias('max_open_min'),
        )
        .toPandas()
    )
    closure['pct_unclosed'] = (
        100.0 * closure['null_clear_dtm'] / closure['n_set'].clip(lower=1)
    ).round(2)
    print(closure.sort_values('fleet').to_string(index=False))
    print()
else:
    print('A.3  SKIPPED: CLEAR_DTM is not in the direct-Silver export projection.')
    print()

# ---------------------------------------------------------------- PART B: outages
print('B.1  device_outage -- the table that actually produces hardware_oos_minutes')
og = read_source(
    'outage',
    ['DEVICE_ID', 'outage_start', 'outage_end', 'duration_min', 'mars_device_category'],
    ['source_event_id', 'is_chargeable', 'failure_level'],
)
og = (
    og
    .withColumn('fleet', canonical_device_category(F.col('mars_device_category')))
    .withColumn('start_ts', F.col('outage_start').cast('timestamp'))
    .withColumn('end_ts', F.col('outage_end').cast('timestamp'))
    .withColumn('dur_min', F.col('duration_min').cast('double'))
    .where(F.col('fleet').isin(*TARGETS))
    .where((F.col('start_ts') >= F.lit(WIN_START)) & (F.col('start_ts') <= F.lit(WIN_END)))
)

DAY = 1440.0
outage_stats = (
    og.groupBy('fleet')
    .agg(
        F.count('*').alias('episodes'),
        F.countDistinct('DEVICE_ID').alias('devices'),
        F.sum(F.when(F.col('end_ts').isNull(), 1).otherwise(0)).alias('null_end'),
        F.sum(F.when(F.col('dur_min').isNull(), 1).otherwise(0)).alias('null_duration'),
        F.sum(F.when(F.col('dur_min') < 0, 1).otherwise(0)).alias('negative_duration'),
        F.sum(F.coalesce('dur_min', F.lit(0.0))).alias('total_min'),
        F.expr('percentile_approx(dur_min, 0.5)').alias('p50_min'),
        F.expr('percentile_approx(dur_min, 0.9)').alias('p90_min'),
        F.expr('percentile_approx(dur_min, 0.99)').alias('p99_min'),
        F.max('dur_min').alias('max_min'),
        F.sum(F.when(F.col('dur_min') > DAY, 1).otherwise(0)).alias('over_1_day'),
        F.sum(F.when(F.col('dur_min') > 7 * DAY, 1).otherwise(0)).alias('over_1_week'),
        F.sum(F.when(F.col('dur_min') > 30 * DAY, 1).otherwise(0)).alias('over_30_days'),
        F.sum(F.when(F.col('dur_min') > DAY, F.col('dur_min')).otherwise(0.0)).alias('min_from_over_1_day'),
    )
    .toPandas()
)
print(outage_stats.sort_values('fleet').to_string(index=False))
print()

# ---------------------------------------------------------------- PART C: rebuild
# Reconstruct the share of available device-hours the same way the dashboard does,
# then remove the long episodes and see what survives. If the share collapses when
# episodes over a day are dropped, the headline is carried by a handful of records
# that never close, not by fleetwide downtime.
print('C.1  Reconstructed share of available device-hours, and what carries it')
rows = []
for r in outage_stats.itertuples(index=False):
    capacity_min = float(r.devices) * float(WIN_DAYS) * DAY
    if capacity_min <= 0:
        continue
    rows.append({
        'fleet': r.fleet,
        'devices': int(r.devices),
        'capacity_device_hours': round(capacity_min / 60.0, 1),
        'recorded_oos_hours': round(float(r.total_min) / 60.0, 1),
        'share_all_episodes_pct': round(100.0 * float(r.total_min) / capacity_min, 2),
        'share_excl_over_1_day_pct': round(
            100.0 * (float(r.total_min) - float(r.min_from_over_1_day)) / capacity_min, 2
        ),
        'episodes_over_1_day': int(r.over_1_day),
        'pct_of_minutes_from_over_1_day': round(
            100.0 * float(r.min_from_over_1_day) / max(float(r.total_min), 1.0), 2
        ),
    })
recon = pd.DataFrame(rows).sort_values('fleet')
print(recon.to_string(index=False))
print()

# ---------------------------------------------------------------- the verdict
print('=' * 78)
print('HOW TO READ THIS')
print('=' * 78)
print(
    'A.2 clear_per_set near 1.0 means episodes close in the event log. Well below 1.0\n'
    '    means Sets are not being cleared, and any duration derived from them is a floor\n'
    '    on nothing -- it is unbounded.\n'
    'A.3 pct_unclosed is the same test per row and is the stronger one. Subtract\n'
    '    clear_after_asof: those are censored by the 11 April boundary, which is expected\n'
    '    and not a defect. What is left is genuinely unclosed.\n'
    'B.1 null_end and the over_1_day / over_1_week / over_30_days counts are where an\n'
    '    impossible share would have to come from. A fare gate cannot be out of service\n'
    '    for 30 continuous days without someone noticing.\n'
    'C.1 is the decisive line. If share_all_episodes_pct is ~95 and\n'
    '    share_excl_over_1_day_pct is small, the headline is an artefact of a few\n'
    '    never-closed episodes and the PS2 availability framing has to be restated.\n'
    '    If the share stays high with long episodes removed, the downtime is real and\n'
    '    broadly distributed, and the finding stands.\n'
)
print(
    'WHAT THIS DOES NOT TEST. It says nothing about data after 11 April 2026. A missing\n'
    'tail can only make a closed window look shorter, never longer, so it cannot be the\n'
    'cause of an inflated share. Sparse Clear events INSIDE the window can be, and that\n'
    'is precisely what A.2 and A.3 measure.\n'
)
